In [145]:
!pip install selenium

^C


In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from datetime import datetime, timedelta
import time
import json
import pandas as pd

# Helpers

In [2]:
def create_driver(remote_url: str = "http://localhost:4444/wd/hub"):
    """Create a Selenium WebDriver connected to the Docker Selenium server."""
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1366,768")
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Remote(
        command_executor=remote_url,
        options=chrome_options
    )
    
    # Execute CDP commands to hide automation
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            })
        """
    })
    
    return driver

In [3]:
def wait_and_click_element_by_text(driver, text: str, timeout: int = 10) -> bool:
    """Wait for an element containing specific text and click it. Returns True if clicked, False if timeout."""
    try:
        element = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((By.XPATH, f"//*[contains(text(), '{text}')]"))
        )
        element.click()
        return True
    except TimeoutException:
        return False

In [4]:
def find_input_by_label_text(driver, label_text: str, input_text: str, timeout: int = 10) -> bool:
    """Find an input field near a label text and type into it. Returns True if successful, False if timeout."""
    try:
        # Find the label element by text
        label_element = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, f"//*[contains(text(), '{label_text}')]"))
        )
        # Navigate to find the associated input (traverse DOM to find input in same container)
        parent = label_element.find_element(By.XPATH, "./ancestor::div[contains(@class, 'css-1dbjc4n')][1]")
        input_element = parent.find_element(By.CSS_SELECTOR, "input")
        input_element.click()
        input_element.clear()
        input_element.send_keys(input_text)
        return True
    except (TimeoutException, NoSuchElementException):
        return False

In [5]:
def get_element_by_data_testid(driver, data_testid: str, timeout: int = 10):
    """Wait for and return an element by data-testid attribute. Returns the element or None if timeout."""
    try:
        element = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, f"[data-testid='{data_testid}']"))
        )
        return element
    except TimeoutException:
        return None

In [6]:
def extract_ticket_info(ticket_container):
    """Extract ticket information from a flight card container element.

    Args:
        ticket_container: Selenium WebElement containing the flight card

    Returns:
        dict: Ticket information or None if extraction fails
    """
    try:
        ticket_info = {}

        # Extract airline logo and name
        airline_img = ticket_container.find_element(By.CSS_SELECTOR, "img[src*='imagekit']")
        ticket_info['airline_logo'] = airline_img.get_attribute('src')

        # Get airline name from the div next to the logo
        airline_name_elem = ticket_container.find_element(
            By.CSS_SELECTOR, "div[dir='auto'].css-cens5h"
        )
        ticket_info['airline_name'] = airline_name_elem.text.strip()

        # Extract baggage allowance - find the svg and get next sibling div
        baggage_elem = ticket_container.find_element(
            By.CSS_SELECTOR, "svg[data-id='IcFacilitiesBaggageEmpty'] + div"
        )
        ticket_info['baggage'] = baggage_elem.text.strip()

        # Extract departure time and airport (first container with time and airport code)
        departure_container = ticket_container.find_element(
            By.CSS_SELECTOR, "div.r-1habvwh.r-eqz5dr.r-9aw3ui.r-knv0ih"
        )
        departure_elems = departure_container.find_elements(By.CSS_SELECTOR, "div[dir='auto']")
        if len(departure_elems) >= 2:
            ticket_info['departure_time'] = departure_elems[0].text.strip()
            ticket_info['departure_airport'] = departure_elems[1].text.strip()

        # Extract arrival time and airport (find container without r-ggk5by class)
        arrival_container = ticket_container.find_element(
            By.CSS_SELECTOR, "div.r-obd0qt.r-eqz5dr.r-9aw3ui.r-knv0ih:not(.r-ggk5by)"
        )
        arrival_elems = arrival_container.find_elements(By.CSS_SELECTOR, "div[dir='auto']")
        if len(arrival_elems) >= 2:
            ticket_info['arrival_time'] = arrival_elems[0].text.strip()
            ticket_info['arrival_airport'] = arrival_elems[1].text.strip()

        # Extract duration
        duration_elem = ticket_container.find_element(
            By.CSS_SELECTOR, "div.r-1awozwy.r-eqz5dr.r-knv0ih.r-bnwqim div[dir='auto']"
        )
        ticket_info['duration'] = duration_elem.text.strip()

        # Extract flight type (Direct/Transit)
        direct_elem = ticket_container.find_element(
            By.CSS_SELECTOR, "div.r-1awozwy.r-6koalj.r-18u37iz.r-kc8jnq div[dir='auto']"
        )
        ticket_info['flight_type'] = direct_elem.text.strip()

        # Extract price - find the h3 with data-testid='label_fl_inventory_price'
        price_elem = ticket_container.find_element(
            By.CSS_SELECTOR, "[data-testid='label_fl_inventory_price']"
        )
        ticket_info['price'] = price_elem.text.strip()

        # Extract original price if exists (strikethrough price in r-142tt33 class)
        try:
            original_price_elem = ticket_container.find_element(
                By.CSS_SELECTOR, "div.r-142tt33[dir='auto']"
            )
            ticket_info['original_price'] = original_price_elem.text.strip()
        except NoSuchElementException:
            ticket_info['original_price'] = None

        # Extract promo labels
        try:
            promo_container = ticket_container.find_element(
                By.CSS_SELECTOR, "[data-testid='view-flight-card-bundle-promo-labels-container']"
            )
            promo_elems = promo_container.find_elements(
                By.CSS_SELECTOR, "[data-testid^='flight-promo-label-pill']"
            )
            ticket_info['promos'] = [promo.text.strip() for promo in promo_elems if promo.text.strip()]
        except NoSuchElementException:
            ticket_info['promos'] = []

        # Extract special tags (e.g., "Bigger Discount") - look for IcSystemTagFill12
        try:
            special_tag = ticket_container.find_element(
                By.CSS_SELECTOR, "svg[data-id='IcSystemTagFill12'] + div"
            )
            ticket_info['special_tag'] = special_tag.text.strip()
        except NoSuchElementException:
            ticket_info['special_tag'] = None

        # Extract "Cheapest" or "Best flight" label if exists
        try:
            best_flight_label = ticket_container.find_element(
                By.CSS_SELECTOR, "div[data-testid='fpr_inventory_card_neonContainerized'] div[dir='auto']"
            )
            ticket_info['highlight_label'] = best_flight_label.text.strip()
        except NoSuchElementException:
            ticket_info['highlight_label'] = None

        return ticket_info

    except (NoSuchElementException, IndexError) as e:
        print(f"Error extracting ticket info: {e}")
        return None


def scroll_and_load_all_tickets(driver, timeout: int = 60, scroll_pause: float = 2.0, num_scrolls: int = 5) -> int:
    """Scroll through the page using the website's internal scroll container.
    
    Uses the Traveloka flight results scroll container and scrolls to bottom
    a specified number of times to load all tickets.
    
    Args:
        driver: Selenium WebDriver
        timeout: Maximum time in seconds to scroll
        scroll_pause: Time to wait between scrolls in seconds
        num_scrolls: Number of times to scroll (default: 5)
        
    Returns:
        int: Number of tickets loaded
    """
    last_count = 0
    no_change_count = 0
    max_no_change = 3  # Stop if no new tickets loaded after this many scrolls
    
    start_time = time.time()
    
    # Find the scrollable container for flight results
    scroll_container = None
    try:
        scroll_container = driver.find_element(
            By.CSS_SELECTOR,
            ".css-1dbjc4n.r-14lw9ot.r-1pi2tsx.r-1rnoaur"
        )
        print("Found scrollable container")
    except NoSuchElementException:
        print("Scroll container not found, using window scroll")
    
    for i in range(num_scrolls):
        # Check if timeout reached
        if time.time() - start_time > timeout:
            print(f"Timeout reached after {timeout} seconds")
            break
        
        if scroll_container:
            # Scroll to bottom of the container
            driver.execute_script("""
                arguments[0].scrollTop = arguments[0].scrollHeight;
            """, scroll_container)
        else:
            # Fallback: scroll the window to bottom
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        
        # Wait for new content to load
        time.sleep(scroll_pause)
        
        # Wait for network to be idle (helps with lazy-loaded content)
        try:
            driver.execute_script("""
                return new Promise((resolve) => {
                    let pending = 0;
                    const origFetch = window.fetch;
                    window.fetch = function(...args) {
                        pending++;
                        return origFetch.apply(this, args).finally(() => {
                            pending--;
                            if (pending === 0) resolve();
                        });
                    };
                    if (pending === 0) resolve();
                    setTimeout(() => resolve(), 1000);
                });
            """)
        except:
            pass  # Continue even if network wait fails
        
        # Count current tickets
        current_tickets = driver.find_elements(
            By.CSS_SELECTOR,
            "[data-testid^='flight-inventory-card-container']"
        )
        current_count = len(current_tickets)
        
        # Check if no new tickets loaded
        if current_count == last_count:
            no_change_count += 1
            if no_change_count >= max_no_change:
                print(f"No new tickets loaded after {no_change_count} attempts")
                break
        else:
            no_change_count = 0
            print(f"Scroll {i+1}/{num_scrolls}: Loaded {current_count} tickets so far...")
        
        last_count = current_count
    
    return len(driver.find_elements(
        By.CSS_SELECTOR,
        "[data-testid^='flight-inventory-card-container']"
    ))


def get_all_tickets(driver, timeout: int = 15, scroll: bool = True, scroll_timeout: int = 60, num_scrolls: int = 5):
    """Get all flight tickets from the search results page.

    Args:
        driver: Selenium WebDriver
        timeout: Maximum time to wait for tickets to load initially
        scroll: Whether to scroll to load more tickets
        scroll_timeout: Maximum time in seconds to scroll for more tickets
        num_scrolls: Number of times to scroll (default: 5)

    Returns:
        list: List of ticket information dictionaries
    """
    try:
        # Wait for flight cards to load - look for the main section
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((
                By.CSS_SELECTOR,
                "[data-testid='view_flight_section_list']"
            ))
        )
        
        # Scroll to load more tickets if enabled
        if scroll:
            print("Scrolling to load all tickets...")
            scroll_and_load_all_tickets(driver, timeout=scroll_timeout, num_scrolls=num_scrolls)
            time.sleep(2)  # Final wait for any lazy-loaded content

        # Find all ticket containers
        ticket_containers = driver.find_elements(
            By.CSS_SELECTOR,
            "[data-testid^='flight-inventory-card-container']"
        )

        tickets = []
        for container in ticket_containers:
            ticket_info = extract_ticket_info(container)
            if ticket_info:
                tickets.append(ticket_info)

        return tickets

    except TimeoutException:
        print("No tickets found within timeout period")
        return []

# Init

In [7]:
driver = create_driver()
base_url = "https://www.traveloka.com/en-id"
driver.get(base_url)

In [8]:
if wait_and_click_element_by_text(driver, "Browse as a guest", timeout=10):
    print("Popup found and clicked 'Browse as a guest'")
else:
    print("Popup not found, skipping...")

Popup found and clicked 'Browse as a guest'


# Destination Input

## Departure

In [9]:
departure_element = get_element_by_data_testid(driver, "airport-autocomplete-container-departure")
departure_element.click()

time.sleep(2)
timeout = 10 
departure_element = WebDriverWait(driver, timeout).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[placeholder='Origin']"))
)
departure_element.send_keys("CGK")

time.sleep(2)
text = "Soekarno Hatta International Airport"
departure_element = WebDriverWait(driver, timeout).until(
    EC.element_to_be_clickable((By.XPATH, f"//*[contains(text(), '{text}')]"))
)
departure_element.click()

## Destination

In [10]:
destination_element = get_element_by_data_testid(driver, "airport-autocomplete-container-destination")
destination_element.click()

time.sleep(2)
timeout = 10 
destination_element = WebDriverWait(driver, timeout).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[placeholder='Destination']"))
)
destination_element.send_keys("DPS")

time.sleep(2)
text = "Ngurah Rai International Airport"
departure_element = WebDriverWait(driver, timeout).until(
    EC.element_to_be_clickable((By.XPATH, f"//*[contains(text(), '{text}')]"))
)
departure_element.click()

# Date Input

In [ ]:
# element = get_element_by_data_testid(driver, "departure-date-input")
# element.click()

# Search Tickets

In [11]:
element = get_element_by_data_testid(driver, "desktop-default-search-button")
element.click()
time.sleep(10)

In [12]:
tickets = get_all_tickets(driver, scroll=True, scroll_timeout=60)

Scrolling to load all tickets...
Found scrollable container
Scroll 1/5: Loaded 17 tickets so far...
Scroll 2/5: Loaded 27 tickets so far...
Scroll 3/5: Loaded 37 tickets so far...
Scroll 4/5: Loaded 47 tickets so far...
Scroll 5/5: Loaded 57 tickets so far...


# Save to CSV

In [13]:
df_tickets = pd.DataFrame(tickets)
df_tickets.to_csv(f"trave_scrap_results_{datetime.now().strftime('%Y%m%d%H%M%S')}.csv")

# Exit Driver

In [14]:
driver.quit()